# Ablation Study — Q-Former Query Tokens (K) and LoRA Rank (r)

Trains several configurations of the VLM on IU-Xray to justify the design
choices (K=64, r=16), saving each checkpoint to Drive and logging to CSV.

**Does:** trains one config at a time, saves checkpoints, logs params + loss.
**Does NOT:** compute NLG/grounding metrics — run your existing eval notebook
on each saved checkpoint for that.

**Budget warning:** each config = one full 3-epoch training run. Start with 2-3
configs, check unit usage, THEN decide whether to run the rest. Trim
`ABLATION_CONFIGS` to fit ~110 Colab units.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 — Imports and device

In [ ]:
import os, gc, time, random
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import pandas as pd
from functools import partial
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torchvision import transforms
from transformers import (AutoImageProcessor, AutoModel, AutoTokenizer,
                          AutoModelForCausalLM, BitsAndBytesConfig,
                          get_cosine_schedule_with_warmup)
from peft import LoraConfig, get_peft_model

cudnn.benchmark = True
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Cell 4 — Ablation configuration

Edit this list to control how many runs you do. **Vary ONE variable at a time.**
Baseline is K=64, r=16. Trim to fit your unit budget — start with 2-3 rows.

In [ ]:
# One-variable-at-a-time ablation grid.
ABLATION_CONFIGS = [
    # Query-token ablation (LoRA rank fixed at 16)
    {"name": "K16_r16", "num_queries": 16, "lora_r": 16},
    {"name": "K32_r16", "num_queries": 32, "lora_r": 16},
    {"name": "K64_r16", "num_queries": 64, "lora_r": 16},   # baseline
    # {"name": "K128_r16", "num_queries": 128, "lora_r": 16},  # only if VRAM allows

    # LoRA-rank ablation (K fixed at 64)
    {"name": "K64_r8",  "num_queries": 64, "lora_r": 8},
    {"name": "K64_r32", "num_queries": 64, "lora_r": 32},
]

EPOCHS = 3
ACCUM_STEPS = 4
BATCH_SIZE = 2
SEED = 42

ABLATION_ROOT = "/content/drive/MyDrive/ablation_study"
WEIGHTS_ROOT  = os.path.join(ABLATION_ROOT, "weights")
RESULTS_CSV   = os.path.join(ABLATION_ROOT, "ablation_results.csv")
os.makedirs(WEIGHTS_ROOT, exist_ok=True)
print("Configs to run:", [c["name"] for c in ABLATION_CONFIGS])

## Cell 5 — Load frozen vision encoder + tokenizer (ONCE, reused for all configs)

In [ ]:
print("Loading Rad-DINO vision encoder...")
vision_processor = AutoImageProcessor.from_pretrained("microsoft/rad-dino")
vision_encoder = AutoModel.from_pretrained("microsoft/rad-dino").to(device)
vision_encoder.eval()
for p in vision_encoder.parameters():
    p.requires_grad = False

LLM_ID = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
tokenizer.pad_token = tokenizer.eos_token
print("Vision encoder + tokenizer ready.")

## Cell 6 — Data pipeline (identical to your training script)

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomAffine(degrees=3, translate=(0.02, 0.02)),
])

class MedicalDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        image = self.dataset[idx]['image']
        if image.mode != 'RGB': image = image.convert('RGB')
        if self.transform: image = self.transform(image)
        return {'image': image, 'report': self.dataset[idx].get('text', "Normal.")}

def get_random_clinical_prompt():
    instructions = [
        "Analyze this chest radiograph and generate a concise clinical report.",
        "Review this CXR and detail findings for the lungs, heart, and pleura.",
        "Provide a structured diagnostic report for this chest X-ray using standard radiology terminology."
    ]
    return ("<|im_start|>system\n"
            "You are an expert radiologist. Write ONLY the diagnostic narrative.<|im_end|>\n"
            f"<|im_start|>user\n{random.choice(instructions)}<|im_end|>\n"
            "<|im_start|>assistant\n")

def collate_fn(batch, processor, tokenizer, absolute_max_length=256):
    images = [b['image'] for b in batch]
    reports = [b['report'] for b in batch]
    pixel_inputs = processor(images=images, return_tensors="pt")
    raw_ids, raw_labels = [], []
    for report in reports:
        p_ids = tokenizer(get_random_clinical_prompt(), add_special_tokens=False).input_ids
        r_ids = tokenizer(report + "<|im_end|>", add_special_tokens=False).input_ids
        raw_ids.append(p_ids + r_ids)
        raw_labels.append([-100]*len(p_ids) + r_ids)
    target_len = min(max(len(x) for x in raw_ids), absolute_max_length)
    bi, bl, bm = [], [], []
    for ids, lbl in zip(raw_ids, raw_labels):
        if len(ids) > target_len:
            ids, lbl = ids[:target_len], lbl[:target_len]
        else:
            pad = target_len - len(ids)
            ids = ids + [tokenizer.pad_token_id]*pad
            lbl = lbl + [-100]*pad
        bi.append(ids); bl.append(lbl)
        bm.append([1 if t != tokenizer.pad_token_id else 0 for t in ids])
    return {'pixel_values': pixel_inputs.pixel_values,
            'input_ids': torch.tensor(bi, dtype=torch.long),
            'labels': torch.tensor(bl, dtype=torch.long),
            'attention_mask': torch.tensor(bm, dtype=torch.long)}

print("Loading IU-Xray dataset...")
full = load_dataset("Shrey-1329/cxiu_hf_dataset", split="train")
tt = full.train_test_split(test_size=0.2, seed=SEED)
train_dataset = MedicalDataset(tt['train'], transform=train_transforms)
collate = partial(collate_fn, processor=vision_processor, tokenizer=tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=2, pin_memory=True, drop_last=True)
print(f"Train batches per epoch: {len(train_loader)}")

## Cell 7 — Model builders (projector parameterised by K; LLM by LoRA rank)

In [ ]:
def build_projector(num_queries, llm_dim, vision_dim=768):
    class QFormerProjector(nn.Module):
        def __init__(self):
            super().__init__()
            self.query_tokens = nn.Parameter(torch.randn(1, num_queries, vision_dim))
            self.cross_attn = nn.MultiheadAttention(vision_dim, num_heads=8, batch_first=True)
            self.norm1 = nn.LayerNorm(vision_dim)
            self.ffn = nn.Sequential(nn.Linear(vision_dim, vision_dim*4), nn.GELU(),
                                     nn.Linear(vision_dim*4, vision_dim))
            self.norm2 = nn.LayerNorm(vision_dim)
            self.proj = nn.Linear(vision_dim, llm_dim)
            self.llm_norm = nn.LayerNorm(llm_dim)
        def forward(self, patch_embeddings):
            sp = patch_embeddings[:, 1:, :]
            q = self.query_tokens.expand(sp.shape[0], -1, -1)
            a, _ = self.cross_attn(q, sp, sp)
            x = self.norm1(q + a)
            x = self.norm2(x + self.ffn(x))
            return self.llm_norm(self.proj(x))
    return QFormerProjector().to(device)

def build_llm(lora_r):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                             bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    base = AutoModelForCausalLM.from_pretrained(LLM_ID, quantization_config=bnb,
                                                device_map={"": 0})
    cfg = LoraConfig(r=lora_r, lora_alpha=lora_r*2,
                     target_modules=["q_proj","k_proj","v_proj","o_proj"],
                     lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
    return get_peft_model(base, cfg)

## Cell 8 — Train-one-config function

In [ ]:
def train_one_config(cfg):
    print(f"\n{'='*55}\nCONFIG {cfg['name']}  (K={cfg['num_queries']}, r={cfg['lora_r']})\n{'='*55}")
    # build LLM first (need hidden size), then projector
    llm = build_llm(cfg["lora_r"])
    projector = build_projector(cfg["num_queries"], llm_dim=llm.config.hidden_size)

    optimizer = torch.optim.AdamW([
        {"params": projector.parameters(), "lr": 1e-4},
        {"params": llm.parameters(), "lr": 2e-5},
    ])
    scaler = torch.amp.GradScaler('cuda')
    total_steps = (len(train_loader)//ACCUM_STEPS)*EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps*0.03), total_steps)

    trainable = (sum(p.numel() for p in projector.parameters() if p.requires_grad)
                 + sum(p.numel() for p in llm.parameters() if p.requires_grad))
    print(f"Trainable params: {trainable/1e6:.2f}M")

    t0 = time.time(); final_loss = None
    for epoch in range(EPOCHS):
        projector.train(); llm.train(); running = 0.0
        for step, batch in enumerate(train_loader):
            pv = batch['pixel_values'].to(device, non_blocking=True)
            ids = batch['input_ids'].to(device, non_blocking=True)
            lbl = batch['labels'].to(device, non_blocking=True)
            am  = batch['attention_mask'].to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                with torch.no_grad():
                    pe = vision_encoder(pv).last_hidden_state.to(torch.float16)
                vp = projector(pe)
                te = llm.get_input_embeddings()(ids)
                emb = torch.cat([vp, te], dim=1)
                b, nvt = ids.shape[0], vp.shape[1]
                vlbl = torch.full((b, nvt), -100, dtype=torch.long, device=device)
                albl = torch.cat([vlbl, lbl], dim=1)
                vam = torch.ones((b, nvt), dtype=torch.long, device=device)
                aam = torch.cat([vam, am], dim=1)
                out = llm(inputs_embeds=emb, attention_mask=aam, labels=albl)
                raw = out.loss.to(torch.float32)
                loss = raw / ACCUM_STEPS
            scaler.scale(loss).backward()
            if (step+1) % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(projector.parameters())+list(llm.parameters()), 1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            running += raw.item()
            if step % 100 == 0:
                print(f"  ep{epoch+1} step{step}/{len(train_loader)} loss={raw.item():.4f}")
        final_loss = running/len(train_loader)
        print(f"  >> epoch {epoch+1} mean loss {final_loss:.4f}")

    elapsed = (time.time()-t0)/60
    save_dir = os.path.join(WEIGHTS_ROOT, cfg["name"])
    os.makedirs(save_dir, exist_ok=True)
    llm.save_pretrained(os.path.join(save_dir, "qwen_lora_adapters"))
    torch.save(projector.state_dict(), os.path.join(save_dir, "vision_projector.pth"))
    print(f"  saved -> {save_dir}  ({elapsed:.1f} min)")

    del projector, llm, optimizer, scaler, scheduler
    gc.collect(); torch.cuda.empty_cache()
    return {"config": cfg["name"], "num_queries": cfg["num_queries"],
            "lora_r": cfg["lora_r"], "trainable_params_M": round(trainable/1e6, 2),
            "final_train_loss": round(final_loss, 4), "minutes": round(elapsed, 1)}

## Cell 9 — Run the ablation sweep

Writes results incrementally and is resume-safe (skips configs already saved),
so a disconnect or quota-out doesn't lose finished runs.
**Watch your unit usage after the first run before continuing.**

In [ ]:
results = []
if os.path.exists(RESULTS_CSV):
    done = pd.read_csv(RESULTS_CSV)
    results = done.to_dict("records")
    done_names = set(done["config"])
    print("Already done:", done_names)
else:
    done_names = set()

for cfg in ABLATION_CONFIGS:
    if cfg["name"] in done_names:
        print(f"Skipping {cfg['name']} (already done)")
        continue
    res = train_one_config(cfg)
    results.append(res)
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
    print(f"  logged -> {RESULTS_CSV}")

print("\nAll requested ablations complete.")
print(pd.DataFrame(results))

## Cell 10 — Next step (evaluation)

Training done. Now run your **existing evaluation notebook** on each saved
checkpoint in `ablation_study/weights/<config_name>/` to get NLG + grounding
metrics per config. Point the eval notebook's `CHECKPOINT_DIR` at each folder,
record METEOR / BERTScore / Pointing Game / IoU, and assemble the final table.

Note: when loading a checkpoint for eval, rebuild the projector with the SAME
`num_queries` and the LLM with the SAME `lora_r` used for that config, or the
`load_state_dict` will fail on shape mismatch.

In [ ]:
print("Saved checkpoints:")
for name in sorted(os.listdir(WEIGHTS_ROOT)):
    print("  ", os.path.join(WEIGHTS_ROOT, name))
print("\nResults CSV:", RESULTS_CSV)
print(pd.read_csv(RESULTS_CSV))